
# Collecte Overpass nationale — expansion multi-secteurs

Ce notebook interroge Overpass API **wilaya par wilaya** (58 wilayas) pour
chaque sous-secteur listé dans `SOUS_SECTEURS_A_COLLECTER`, et insère
directement dans `entreprises` avec `wilaya_name` déjà connu (puisqu'on
interroge par wilaya, on n'a plus besoin de matcher une wilaya a posteriori
comme pour le lot déjà en base).

**Deux changements de fond par rapport à la collecte précédente :**

1. **Dédup par `source_id` (id OSM natif), plus par nom.** Le bug de
   contrainte UNIQUE rencontré sur les établissements sans nom propre
   ("Cabinet médical" x6 dans une même commune) ne peut plus se reproduire :
   chaque élément OSM a un id unique (`node/12345`, `way/67890`), utilisé
   comme clé de dédup avant insertion.
2. **`wilaya_name` connu dès la collecte**, plus de risque de `NULL` massif
   comme sur `cabinet médical`/`laboratoire d'analyses`.

**Ce que ce notebook NE couvre PAS** (pas d'équivalent OSM fiable) :
- Éditeurs de logiciels de santé
- Apporteurs d'affaires

Ces deux-là nécessitent une recherche web ciblée (LinkedIn, annuaires
professionnels IT), pas du scraping géographique — à traiter séparément,
probablement en adaptant le `DiscoveryAgent` existant plutôt qu'Overpass.

**Avant de lancer le pipeline de scoring (`main.py`) sur les nouveaux
secteurs** (`assurance`, `juridique`, `industrie`, `étatique`), il faudra
ajouter une entrée correspondante dans `config.SECTOR_CONFIGS` — sinon
`ScoringAgent` lèvera une `ValueError` (`Aucune configuration de scoring
pour le secteur...`).


In [1]:
from pathlib import Path
import os

print("CWD =", os.getcwd())

print("\n--- backend ---")
print(Path.cwd() / "data")
print("data existe :", (Path.cwd() / "data").exists())

print("\n--- DB avec data/... ---")
p1 = Path("data/dasec_prospection.db")
print("chemin :", p1.resolve())
print("existe :", p1.exists())
print("fichier :", p1.is_file())

print("\n--- DB avec backend/data/... ---")
p2 = Path("backend/data/dasec_prospection.db")
print("chemin :", p2.resolve())
print("existe :", p2.exists())
print("fichier :", p2.is_file())

CWD = c:\Users\HP\Downloads\ProspectionAI\backend\notebooks

--- backend ---
c:\Users\HP\Downloads\ProspectionAI\backend\notebooks\data
data existe : False

--- DB avec data/... ---
chemin : C:\Users\HP\Downloads\ProspectionAI\backend\notebooks\data\dasec_prospection.db
existe : False
fichier : False

--- DB avec backend/data/... ---
chemin : C:\Users\HP\Downloads\ProspectionAI\backend\notebooks\backend\data\dasec_prospection.db
existe : False
fichier : False


In [1]:
import sqlite3
import time
import unicodedata
import difflib
import requests

DB_PATH = "../data/dasec_prospection.db"

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
DELAI_ENTRE_REQUETES_SEC = 1.5
TIMEOUT_REQUETE_SEC = 90

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

print("Connecté à", DB_PATH)

Connecté à ../data/dasec_prospection.db


## 1. Wilayas et communes de référence (depuis la DB existante)

In [2]:

# On construit la liste des wilayas et un index des communes par wilaya
# directement depuis la table `communes` déjà en base — pas de liste en dur,
# donc pas de désynchronisation possible avec la source de vérité existante.

cursor = conn.execute("SELECT DISTINCT wilaya_code, wilaya_name FROM communes WHERE wilaya_code IS NOT NULL")
WILAYAS = sorted(
    [(row["wilaya_code"], row["wilaya_name"]) for row in cursor.fetchall()],
    key=lambda t: int(t[0])
)
print(f"{len(WILAYAS)} wilayas trouvées.")

def iso_code(wilaya_code: str) -> str:
    return f"DZ-{int(wilaya_code):02d}"

# Index des communes par wilaya_code pour un matching texte rapide (fuzzy)
COMMUNES_PAR_WILAYA = {}
cursor = conn.execute("SELECT id, commune_name, wilaya_code FROM communes")
for row in cursor.fetchall():
    COMMUNES_PAR_WILAYA.setdefault(row["wilaya_code"], []).append((row["id"], row["commune_name"] or ""))

def normaliser(texte: str) -> str:
    if not texte:
        return ""
    texte = unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode()
    return texte.lower().strip()

def matcher_commune(wilaya_code: str, texte_brut: str):
    """Retourne (commune_id, commune_name) le plus proche, ou (None, None).
    Fuzzy match limité à la wilaya déjà connue -> peu de faux positifs."""
    if not texte_brut:
        return None, None
    candidats = COMMUNES_PAR_WILAYA.get(wilaya_code, [])
    if not candidats:
        return None, None
    cible = normaliser(texte_brut)
    noms_normalises = {normaliser(nom): (cid, nom) for cid, nom in candidats}
    proches = difflib.get_close_matches(cible, noms_normalises.keys(), n=1, cutoff=0.72)
    if proches:
        return noms_normalises[proches[0]]
    return None, None


58 wilayas trouvées.


## 2. Config des sous-secteurs à collecter (tags Overpass)

In [3]:

def classifier_hopital(nom: str) -> str:
    n = normaliser(nom)
    if "chu" in n:
        return "CHU"
    if "ehs" in n:
        return "EHS"
    if "eph" in n or "etablissement public hospitalier" in n:
        return "EPH"
    return "hôpital"

# Chaque entrée : secteur, sous_secteur (ou classifier() pour sous-catégoriser
# après coup), et une liste de requêtes Overpass QL (node + way).
SOUS_SECTEURS_A_COLLECTER = [
    {
        "secteur": "santé", "sous_secteur": None, "classifier": classifier_hopital,
        "filtres": ['node["amenity"="hospital"]'],  # séparé du way
    },
    {
        "secteur": "santé", "sous_secteur": None, "classifier": classifier_hopital,
        "filtres": ['way["amenity"="hospital"]'],
    },
    {
        "secteur": "santé", "sous_secteur": "clinique privée",
        "filtres": ['node["healthcare"="clinic"]', 'way["healthcare"="clinic"]',
                    'node["amenity"="clinic"]', 'way["amenity"="clinic"]'],
    },
    {
        "secteur": "santé", "sous_secteur": "laboratoire d'analyses",
        "filtres": ['node["healthcare"="laboratory"]', 'way["healthcare"="laboratory"]'],
    },
    {
        "secteur": "santé", "sous_secteur": "centre de transfusion sanguine",
        "filtres": ['node["healthcare"="blood_donation"]', 'way["healthcare"="blood_donation"]',
                    'nwr["name"~"transfusion",i]'],
    },
    {
        "secteur": "santé", "sous_secteur": "opticien",
        "filtres": ['node["shop"="optician"]', 'way["shop"="optician"]'],
    },
    {
        "secteur": "santé", "sous_secteur": "cabinet dentaire",
        "filtres": ['node["amenity"="dentist"]', 'way["amenity"="dentist"]',
                    'node["healthcare"="dentist"]'],
    },
    {
        "secteur": "santé", "sous_secteur": "cabinet médical",
        "filtres": ['node["amenity"="doctors"]', 'way["amenity"="doctors"]',
                    'node["healthcare"="doctor"]'],
    },
    {
        "secteur": "assurance", "sous_secteur": "assurance privée",
        "filtres": ['node["office"="insurance"]', 'way["office"="insurance"]'],
    },
    {
        "secteur": "assurance", "sous_secteur": "CNAS",
        "filtres": ['nwr["name"~"CNAS",i]'],
    },
    {
        "secteur": "assurance", "sous_secteur": "CASNOS",
        "filtres": ['nwr["name"~"CASNOS",i]'],
    },
    {
        "secteur": "juridique", "sous_secteur": "cabinet d'avocat",
        "filtres": ['node["office"="lawyer"]', 'way["office"="lawyer"]'],
    },
    {
        "secteur": "juridique", "sous_secteur": "cabinet comptable",
        "filtres": ['node["office"="accountant"]', 'way["office"="accountant"]'],
    },
    {
        "secteur": "juridique", "sous_secteur": "notaire",
        "filtres": ['node["office"="notary"]', 'way["office"="notary"]'],
    },
    {
        # Tag bruyant par nature (beaucoup de faux positifs OSM sur `craft`) :
        # à trier/filtrer après import plutôt qu'à la collecte.
        "secteur": "industrie", "sous_secteur": "entreprise industrielle",
        "filtres": ['node["building"="industrial"]["name"]', 'way["building"="industrial"]["name"]'],
    },
    {
        "secteur": "étatique", "sous_secteur": "mairie",
        "filtres": ['node["amenity"="townhall"]', 'way["amenity"="townhall"]'],
    },
    {
        "secteur": "étatique", "sous_secteur": "siège de wilaya",
        "filtres": ['nwr["name"~"Wilaya de",i]'],
    },
]

print(f"{len(SOUS_SECTEURS_A_COLLECTER)} groupes de collecte configurés.")


17 groupes de collecte configurés.


## 3. Requête Overpass par wilaya

In [ ]:

def construire_requete(wilaya_iso: str, filtres: list[str]) -> str:
    corps = "\n".join(f'  {f}(area.zone);' for f in filtres)
    return f"""
[out:json][timeout:180];
area["ISO3166-2"="{wilaya_iso}"]->.zone;
(
{corps}
);
out center tags;
"""

def interroger_overpass(
    wilaya_iso: str,
    filtres: list[str],
    essais: int = 3
):
    requete = construire_requete(wilaya_iso, filtres)

    headers = {
        "User-Agent": "ProspectionAI/1.0 (contact: votre-email@example.com)",
        "Accept": "application/json",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    }

    for tentative in range(1, essais + 1):
        try:
            reponse = requests.post(
                OVERPASS_URL,
                data={"data": requete},
                headers=headers,
                timeout=TIMEOUT_REQUETE_SEC
            )

            if reponse.status_code == 200:
                return reponse.json().get("elements", [])

            if reponse.status_code == 429:
                print(
                    f"  [Overpass] rate-limited, pause 30s "
                    f"(tentative {tentative}/{essais})"
                )
                time.sleep(30)
                continue

            print(
                f"  [Overpass] code HTTP {reponse.status_code}, "
                f"tentative {tentative}/{essais}"
            )
            print("  Réponse :", reponse.text[:500])

        except requests.exceptions.RequestException as e:
            print(
                f"  [Overpass] erreur réseau : "
                f"{type(e).__name__}: {e}, "
                f"tentative {tentative}/{essais}"
            )

        time.sleep(5 * tentative)

    return []


## 4. Extraction des champs + insertion dédupliquée

In [5]:

def extraire_champs(element: dict) -> dict:
    tags = element.get("tags", {})
    lat = element.get("lat") or (element.get("center") or {}).get("lat")
    lon = element.get("lon") or (element.get("center") or {}).get("lon")
    return {
        "source_id": f'{element["type"]}/{element["id"]}',
        "nom": tags.get("name"),
        "telephone": tags.get("contact:phone") or tags.get("phone"),
        "email": tags.get("contact:email") or tags.get("email"),
        "site_web": tags.get("contact:website") or tags.get("website"),
        "facebook": tags.get("contact:facebook"),
        "latitude": lat,
        "longitude": lon,
        "commune_brute": tags.get("addr:city") or tags.get("addr:municipality") or tags.get("is_in"),
        "adresse": tags.get("addr:full") or tags.get("addr:street"),
    }

def deja_importe(source_id: str) -> bool:
    row = conn.execute(
        "SELECT id FROM entreprises WHERE source_id = ?", (source_id,)
    ).fetchone()
    return row is not None

def inserer_etablissement(champs: dict, secteur: str, sous_secteur: str, wilaya_code: str, wilaya_name: str):
    if deja_importe(champs["source_id"]):
        return False

    nom = champs["nom"] or f"{sous_secteur} (sans nom)"
    commune_id, commune_name = matcher_commune(wilaya_code, champs["commune_brute"])

    valeurs = (
        nom, secteur, sous_secteur, champs["telephone"], champs["email"],
        champs["site_web"], champs["facebook"], champs["latitude"], champs["longitude"],
        commune_id, commune_name or champs["commune_brute"], wilaya_name,
        champs["adresse"], champs["source_id"],
    )

    try:
        conn.execute("""
            INSERT INTO entreprises (
                nom, secteur, sous_secteur, telephone, email, site_web, facebook,
                latitude, longitude, commune_id, commune_brute, wilaya_name,
                adresse, source, source_id
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 'osm', ?)
        """, valeurs)
        return True
    except sqlite3.IntegrityError:
        source_unique = f"osm#{champs['source_id']}"
        try:
            conn.execute("""
                INSERT INTO entreprises (
                    nom, secteur, sous_secteur, telephone, email, site_web, facebook,
                    latitude, longitude, commune_id, commune_brute, wilaya_name,
                    adresse, source, source_id
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, valeurs[:-1] + (source_unique, champs["source_id"]))
            return True
        except sqlite3.IntegrityError as e:
            # Cas résiduel imprévu : on log et on continue plutôt que de
            # planter toute la boucle sur un seul établissement.
            print(f"  [ignoré définitivement] {champs['source_id']} : {e}")
            return False

In [6]:
OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

DELAI_ENTRE_REQUETES_SEC = 4  # augmenté depuis 1.5

def interroger_overpass(wilaya_iso: str, filtres: list[str], essais: int = 4):
    requete = construire_requete(wilaya_iso, filtres)
    headers = {"User-Agent": "ProspectionAI/1.0 (contact: votre-email@example.com)"}

    for tentative in range(1, essais + 1):
        url = OVERPASS_MIRRORS[(tentative - 1) % len(OVERPASS_MIRRORS)]
        try:
            reponse = requests.post(url, data={"data": requete}, headers=headers, timeout=120)
            if reponse.status_code == 200:
                return reponse.json().get("elements", [])
            if reponse.status_code in (429, 406):
                pause = 45 * tentative  # backoff croissant : 45s, 90s, 135s...
                print(f"  [Overpass] limité ({reponse.status_code}) sur {url}, pause {pause}s")
                time.sleep(pause)
                continue
            print(f"  [Overpass] code HTTP {reponse.status_code} sur {url}, tentative {tentative}/{essais}")
        except requests.exceptions.RequestException as e:
            print(f"  [Overpass] erreur réseau sur {url} : {type(e).__name__}, tentative {tentative}/{essais}")
        time.sleep(10 * tentative)
    return []

In [8]:
test = requests.post(
    "https://overpass-api.de/api/interpreter",
    data={"data": '[out:json][timeout:180];area["ISO3166-2"="DZ-16"]->.zone;(node["amenity"="hospital"](area.zone););out center tags;'},
    headers={"User-Agent": "ProspectionAI/1.0"},
    timeout=30,
)
print(test.status_code, test.text[:200])

200 {
  "version": 0.6,
  "generator": "Overpass API 0.7.62.11 87bfad18",
  "osm3s": {
    "timestamp_osm_base": "2026-08-09T09:22:21Z",
    "timestamp_areas_base": "2026-08-08T01:53:06Z",
    "copyright"


In [9]:
tentative_retry = 0
while echecs and tentative_retry < 3:
    tentative_retry += 1
    print(f"\n--- Retry {tentative_retry} sur {len(echecs)} échecs ---")
    echecs_restants = []
    for wilaya_code, wilaya_name, label in echecs:
        groupe = next(g for g in SOUS_SECTEURS_A_COLLECTER if (g["sous_secteur"] or f"{g['secteur']} (classifié après coup)") == label)
        wilaya_iso = iso_code(wilaya_code)
        elements = interroger_overpass(wilaya_iso, groupe["filtres"])
        if not elements:
            echecs_restants.append((wilaya_code, wilaya_name, label))
            continue
        for element in elements:
            champs = extraire_champs(element)
            sous_secteur = groupe["classifier"](champs["nom"]) if groupe.get("classifier") else groupe["sous_secteur"]
            inserer_etablissement(champs, groupe["secteur"], sous_secteur, wilaya_code, wilaya_name)
        conn.commit()
        time.sleep(DELAI_ENTRE_REQUETES_SEC)
    echecs = echecs_restants
    if echecs:
        print(f"  Pause 60s avant prochain retry ({len(echecs)} restants)...")
        time.sleep(60)

print(f"\nÉchecs définitifs après retries : {len(echecs)}")

NameError: name 'echecs' is not defined

## 5. Boucle principale — attention, potentiellement longue (58 wilayas × 16 groupes)

In [ ]:

resume = {}  # (secteur, sous_secteur_ou_None) -> nb insérés

for groupe in SOUS_SECTEURS_A_COLLECTER:
    secteur = groupe["secteur"]
    sous_secteur_fixe = groupe["sous_secteur"]
    classifier = groupe.get("classifier")
    label = sous_secteur_fixe or f"{secteur} (classifié après coup)"
    print(f"\\n=== {label} ===")

    inseres_ce_groupe = 0

    for wilaya_code, wilaya_name in WILAYAS:
        wilaya_iso = iso_code(wilaya_code)
        elements = interroger_overpass(wilaya_iso, groupe["filtres"])

        for element in elements:
           champs = extraire_champs(element)
           sous_secteur = classifier(champs["nom"]) if classifier else sous_secteur_fixe
           if inserer_etablissement(champs, secteur, sous_secteur, wilaya_code, wilaya_name):
              inseres_ce_groupe += 1
        conn.commit()
        time.sleep(DELAI_ENTRE_REQUETES_SEC)

    print(f"  -> {inseres_ce_groupe} nouveaux établissements insérés pour ce groupe.")
    resume[label] = inseres_ce_groupe

print("\\n=== Résumé global ===")
for label, n in resume.items():
    print(f"{label:45} {n:5} insérés")


\n=== santé (classifié après coup) ===
  [Overpass] code HTTP 504 sur https://overpass-api.de/api/interpreter, tentative 1/4


## 6. Vérification post-import

In [ ]:

cursor = conn.execute(\"\"\"
    SELECT secteur, sous_secteur, COUNT(DISTINCT wilaya_name) as nb_wilayas, COUNT(*) as total
    FROM entreprises
    WHERE source = 'osm'
    GROUP BY secteur, sous_secteur
    ORDER BY nb_wilayas ASC
\"\"\")
for row in cursor.fetchall():
    print(f"{row['secteur']:12} {row['sous_secteur'] or '—':30} {row['nb_wilayas']:3} wilayas   {row['total']:5} total")

conn.close()
